# 03 Discrete Continuous DP

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE)
[![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)

In [ ]:
# === Environment Setup ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize_scalar
from IPython.display import display, Markdown

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (11, 7), 'figure.dpi': 130})
np.set_printoptions(suppress=True, linewidth=120, precision=4)

print("Environment initialized.")

## Part 3: Dynamic Models
## Chapter 3.3: Discrete-Continuous Dynamic Programming

### Table of Contents
1.  [Discrete-Continuous Choice Problems](#1.-Discrete-Continuous-Choice-Problems)
2.  [The Rust (1987) Model](#2.-The-Rust-(1987)-Model)
3.  [Implementing the Nested Fixed Point Algorithm](#3.-Implementing-the-Nested-Fixed-Point-Algorithm)
4.  [Summary](#4.-Summary)

# The Lens: Mixed Decisions

**What problem are we solving?**
Many economic problems involve both **discrete choices** (e.g., work/retire, buy/rent, default/repay) and **continuous choices** (e.g., how much to consume, how much to save). These "discrete-continuous" problems are ubiquitous in labor economics and corporate finance.

**Why this method?**
The presence of discrete choices introduces "kinks" or non-concavities in the value function, breaking standard derivative-based optimization methods. We need a specialized approach:
1.  **Conditional Value Functions:** Calculate the value of *each* discrete choice (e.g., $V_{work}$ and $V_{retire}$) assuming optimal continuous choices.
2.  **Upper Envelope:** The true value function is the maximum of these conditional values: $V(x) = \max(V_{work}(x), V_{retire}(x))$.

This notebook demonstrates how to handle these non-standard but critical optimization landscapes.

### 1. Discrete-Continuous Choice Problems

Many important economic problems involve agents making both discrete and continuous choices. For example:
- A consumer decides **whether** to buy a new car (discrete) and **how much** to spend on other goods (continuous).
- A firm decides **whether** to replace a machine (discrete) and **how much** to spend on its maintenance (continuous).
- An individual decides **whether** to work (discrete) and **how many hours** to supply (continuous).

These problems are challenging because the value function is often not concave, meaning standard gradient-based optimizers can get stuck in local optima. The solution strategy typically involves computing **conditional value functions** for each discrete choice.

### 2. The Rust (1987) Model: A Canonical Example

Rust's model provides a canonical example of a discrete-continuous DP problem. A manager, Harold Zurcher, must decide each period whether to replace a bus engine (`d=1`, a discrete choice) or to keep it and perform maintenance (`d=0`).

- **State Variable:** The state of the system is the odometer reading, `x`, which represents the engine's mileage.
- **Flow Utility:** The utility in a period depends on the maintenance cost, which is a function of mileage, and the operating costs.
- **Transition:** The mileage `x` evolves stochastically over time.

The manager's problem is to choose a sequence of replacement and maintenance decisions to minimize the total expected discounted cost over the infinite horizon.

#### The Conditional Value Function

The key to solving these models is to decompose the value function. The overall value function is:
$$ V(x) = \max_{d \in \{0, 1\}} \{ v(x, d) \} $$
where $v(x, d)$ is the **conditional value function**—the value of committing to the discrete choice `d`.

1.  **Value of Replacing (`d=1`):** If the manager replaces the engine, he pays a fixed replacement cost `RC` and gets a new engine with zero mileage. The value is:
    $$ v(x, 1) = -RC + \beta E[V(0)] $$

2.  **Value of Keeping (`d=0`):** If the manager keeps the engine, he pays maintenance cost $c(x)$ and moves to a new state $x'$.
    $$ v(x, 0) = -c(x) + \beta E[V(x') | x] $$

This structure creates a **nested fixed point problem**. To solve for the outer value function $V(x)$, we iterate on these conditional value functions.

In [ ]:
### Implementing the Rust Model

class OptimalReplacement:
    """
    A class to solve the Rust (1987) optimal replacement problem using 
    Value Function Iteration.
    """
    def __init__(self, beta=0.9, RC=10, c_scale=0.01, max_mileage=100, n_states=100):
        self.beta = beta          # Discount factor
        self.RC = RC              # Replacement cost
        self.c_scale = c_scale    # Cost scale parameter
        self.n_states = n_states  # Number of grid points
        self.state_grid = np.arange(n_states)
        
        # Transition probability: mileage increases by 0, 1, or 2 units
        self.p = [0.1, 0.7, 0.2]
        self.P = self._build_transition_matrix()
        
        # Cost function: linear in mileage
        self.cost = self.c_scale * self.state_grid

    def _build_transition_matrix(self):
        """Constructs the transition matrix P[x, x']."""
        P = np.zeros((self.n_states, self.n_states))
        for i in range(self.n_states):
            for k, prob in enumerate(self.p):
                if i + k < self.n_states:
                    P[i, i+k] = prob
            # Absorbing state at the boundary
            P[i, -1] += 1 - np.sum(P[i, :])
        return P

    def solve(self, tol=1e-6, max_iter=1000):
        """Solves the model using Value Function Iteration."""
        V = np.zeros(self.n_states)
        
        for i in range(max_iter):
            # 1. Expected Value of future state
            EV = self.P @ V
            
            # 2. Conditional Value Functions
            # Value of Keeping (d=0)
            v_keep = -self.cost + self.beta * EV
            
            # Value of Replacing (d=1)
            # If replaced, state resets to 0. Expected value is V[0].
            # But strictly, in Rust's model, you replace and THEN transitions happen from 0.
            # Simplified: You replace, cost is RC, and you start next period at state 0.
            v_replace = -self.RC + self.beta * V[0]
            
            # 3. Bellman Operator
            V_new = np.maximum(v_keep, v_replace)
            
            if np.max(np.abs(V_new - V)) < tol:
                print(f"Converged in {i} iterations.")
                return V_new, v_keep, v_replace
            V = V_new
            
        return V, v_keep, v_replace

model = OptimalReplacement(RC=8)
V, v_keep, v_replace = model.solve()

# Find the threshold state where optimal choice switches from Keep to Replace
policy = (v_replace > v_keep).astype(int)
threshold_state = np.argmax(policy)
print(f"Optimal Replacement Threshold: State {threshold_state} (Mileage {threshold_state})")

In [ ]:
### Visualizing the Optimal Policy

plt.figure(figsize=(10, 6))
plt.plot(model.state_grid, v_keep, label='Value of Keeping Engine', linewidth=2)
plt.plot(model.state_grid, np.full_like(model.state_grid, v_replace), label='Value of Replacing', linestyle='--', linewidth=2)
plt.axvline(x=threshold_state, color='red', linestyle=':', label=f'Replacement Threshold (x={threshold_state})')

plt.title('Optimal Replacement Policy: Keep vs. Replace')
plt.xlabel('Mileage (State x)')
plt.ylabel('Value Function')
plt.legend()
plt.show()

# Summary

Discrete-continuous problems model the most important decisions in life: career, housing, and default.

**Key Takeaways:**
*   **Conditional Maximization:** Solve the continuous problem for *each* discrete option first, then compare them.
*   **Non-Concavity:** Discrete choices destroy the global concavity of the value function. We must be careful with local optimizers.
*   **Thresholds:** The solution is often characterized by threshold values (e.g., a reservation wage or a default threshold) where the optimal discrete choice switches.